In [1]:
import sys; sys.path.append("..")
import pandas as pd
from src.evaluation import evalue, affiche
from src.modeles import regression_logistique
from src.graines import set_seed
from src.config import PREPARE

set_seed()
X_app = pd.read_csv(PREPARE / "X_app_final.csv", index_col=0)
y_app = pd.read_csv(PREPARE / "y_app.csv", index_col=0).squeeze()

# témoin : un modèle constant, pour vérifier que la chaîne tourne
from sklearn.dummy import DummyClassifier
r0 = evalue(lambda: DummyClassifier(strategy="prior"), X_app, y_app, "témoin")
print(r0[["pli", "cout", "rappel", "FN"]].to_string(index=False))


2026-08-22 16:06:50.963518: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-22 16:06:51.012605: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


 pli  cout  rappel  FN
   1 80000     0.0 160
   2 80000     0.0 160
   3 80000     0.0 160
   4 80000     0.0 160
   5 80000     0.0 160


In [2]:
# régression logistique, la même grille qu'avant
resultats = {}
for C in [0.01, 0.1, 1.0, 10.0]:
    r = evalue(lambda C=C: regression_logistique(C), X_app, y_app, f"RL C={C}")
    resultats[C] = r
    print(f"C={C:<6} coût {r['cout'].mean():>9,.0f}  ± {r['cout'].std():>7,.0f}")

meilleur_C = min(resultats, key=lambda c: resultats[c]["cout"].mean())
affiche(resultats[meilleur_C], f"Régression logistique, C = {meilleur_C}")

C=0.01   coût     9,596  ±   1,515
C=0.1    coût    10,196  ±     646
C=1.0    coût    10,502  ±     901
C=10.0   coût    10,960  ±   1,017
  Régression logistique, C = 0.01
  COÛT MOYEN :      9,596  ±  1,515
  échelle d'un pli : 9 600 lignes, ~160 pannes
  règle naïve sur un pli : 80 000
  seuil dépondéré moyen : 0.0128  (repère 0,0196)

                   moyenne  ecart_type
cout             9596.0000   1515.0677
rappel              0.9212      0.0236
precision           0.3186      0.0595
auc                 0.9780      0.0118
auc_pr              0.7512      0.0121
VP                147.4000      3.7815
FP                329.6000     94.6007
FN                 12.6000      3.7815
seuil               0.3840      0.0904
seuil_depondere     0.0128      0.0046


In [1]:
import sys; sys.path.append("..")
import pandas as pd
from src.config import PREPARE
from src.evaluation import evalue, affiche, resume
from src.graines import set_seed
import src.modeles as M

set_seed()
X_app = pd.read_csv(PREPARE / "X_app_final.csv", index_col=0)
y_app = pd.read_csv(PREPARE / "y_app.csv", index_col=0).squeeze()
tableau = {}


2026-08-24 15:22:49.233557: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-24 15:22:49.281744: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
rf_hors = evalue(lambda: M.foret_aleatoire(), X_app, y_app, "RF")
rf_int  = evalue(lambda: M.foret_aleatoire(), X_app, y_app, "RF interne",
                 seuil_hors_echantillon=False)

for nom, r in [("seuil hors échantillon", rf_hors), ("seuil interne", rf_int)]:
    print(f"{nom:<24} coût {r['cout'].mean():>9,.0f} ± {r['cout'].std():>7,.0f}"
          f"   seuil {r['seuil'].mean():.4f}   FN {r['FN'].mean():.1f}")

seuil hors échantillon   coût     6,714 ±     530   seuil 0.0320   FN 5.8
seuil interne            coût    41,050 ±   2,557   seuil 0.5653   FN 82.0


In [4]:
for n, prof, feuille in [(300, None, 1), (300, 20, 1), (300, None, 5), (500, None, 1)]:
    r = evalue(lambda n=n, p=prof, f=feuille: M.foret_aleatoire(n, p, f),
               X_app, y_app, f"RF {n}/{prof}/{feuille}")
    print(f"n={n} prof={prof} feuille={feuille} : "
          f"{r['cout'].mean():>9,.0f} ± {r['cout'].std():>7,.0f}")
    tableau[f"RF {n}/{prof}/{feuille}"] = r

n=300 prof=None feuille=1 :     6,714 ±     530
n=300 prof=20 feuille=1 :     7,170 ±     847
n=300 prof=None feuille=5 :     7,220 ±   1,022
n=500 prof=None feuille=1 :     6,796 ±   1,102


In [5]:
for prof, taux in [(4, 0.1), (6, 0.1), (6, 0.05), (8, 0.1)]:
    r = evalue(lambda p=prof, t=taux: M.xgboost(p, t), X_app, y_app,
               f"XGB {prof}/{taux}")
    print(f"prof={prof} taux={taux} : {r['cout'].mean():>9,.0f} ± {r['cout'].std():>7,.0f}")
    tableau[f"XGB {prof}/{taux}"] = r

prof=4 taux=0.1 :     7,636 ±   1,224
prof=6 taux=0.1 :     7,074 ±   1,499
prof=6 taux=0.05 :     6,960 ±   1,044
prof=8 taux=0.1 :     6,576 ±   1,030


In [2]:
for prof, taux in [(4, 0.1), (6, 0.1), (6, 0.05), (8, 0.1)]:
    r = evalue(lambda p=prof, t=taux: M.xgboost(p, t), X_app, y_app,
               f"XGB {prof}/{taux}")
    print(f"prof={prof} taux={taux} : {r['cout'].mean():>9,.0f} ± {r['cout'].std():>7,.0f}")
    tableau[f"XGB {prof}/{taux}"] = r


prof=4 taux=0.1 :     7,636 ±   1,224
prof=6 taux=0.1 :     7,074 ±   1,499
prof=6 taux=0.05 :     6,960 ±   1,044


KeyboardInterrupt: 

In [ ]:
for C in [0.001, 0.01, 0.1]:
    r = evalue(lambda C=C: M.svm_lineaire(C), X_app, y_app, f"SVM lin C={C}")
    print(f"C={C} : {r['cout'].mean():>9,.0f} ± {r['cout'].std():>7,.0f}")
    tableau[f"SVM lin C={C}"] = r

In [ ]:
for couches in [(64, 32), (128, 64), (256, 128, 64)]:
    r = evalue(lambda c=couches: M.PerceptronKeras(couches=c), X_app, y_app,
               f"MLP {couches}", n_jobs=1)
    print(f"{couches} : {r['cout'].mean():>9,.0f} ± {r['cout'].std():>7,.0f}")
    tableau[f"MLP {couches}"] = r

In [ ]:
synthese = pd.DataFrame({
    nom: {"cout": r["cout"].mean(), "ecart_type": r["cout"].std(),
          "rappel": r["rappel"].mean(), "precision": r["precision"].mean(),
          "FN": r["FN"].mean(), "FP": r["FP"].mean(),
          "auc_pr": r["auc_pr"].mean(),
          "seuil_dep": r["seuil_depondere"].mean()}
    for nom, r in tableau.items()}).T.sort_values("cout").round(4)

synthese.to_csv("../reports/benchmark.csv")
print(synthese.to_string())